# 🚀 LLM Evaluation und Prompt-Tuning

> Willkommen! In diesem Notebook lernst du:
> 1. Wie man ein LLM aufruft
> 2. Dass LLMs **nicht perfekt** sind
> 3. Wie man **Accuracy misst**
> 4. Wie du **selbst Prompts verbesserst** — und warum das mühsam ist
>
# Wie navigiere ich durch dieses Notebook?
> 1. Lies was steht
> 2. Drücke Shift-Enter um den Code auszuführen.
> 3. Experimentiere mit dem Prompts und den Daten.
>
# Häufige Fehler
> Die Kasten müssen von oben nach unten ausgeführt werden.
> Zuerst musst du einmal das ./start.sh laufen lassen.


## Worum geht's hier?

Dieses Notebook hat zwei Teile:

1. **Teil 1** — Wir rufen ein LLM auf, sehen wo es Fehler macht, und lernen 5 Strategien dagegen
2. **Teil 2** — Wir messen Qualität mit Metriken und tunen Prompts — erst manuell, dann mit Benchmarks

Am Ende weisst du: Wie gut ist mein Modell? Und wie mache ich es besser?

> **Tipp:** Jede Code-Zelle kannst du mit Shift+Enter ausführen. Die Ergebnisse erscheinen direkt darunter.


## 🛠️ Setup

Wir importieren die nötigen Bibliotheken und unsere Task-Bibliothek.
- **Tasks**: 20 kuratierte Aufgaben in 4 Schwierigkeitsstufen
- **Visualisierungen**: Interaktive Widgets zum Experimentieren
- **Actions**: Funktionen die LLMs aufrufen und Ergebnisse auswerten

In [ ]:
import sys
sys.path.insert(0, ".")  # notebooks/ is the working dir
import dspy
from dspy_tasks.tasks import list_tasks, task_summary, get_task
from dspy_tasks.visualize import display_score, display_insight, display_tier_header

## 🗺️ Die Reise durch alle Notebooks

Hier siehst du die fünf Stationen auf einen Blick. Wir starten mit **Evaluation** — also der Frage: *Wie gut ist mein Modell eigentlich?*

Jedes Notebook baut auf dem vorherigen auf. Am Ende hast du ein komplettes Bild davon, wie man KI-Systeme systematisch verbessert.


In [ ]:
from dspy_tasks.visualize import diagram
diagram([
    {"label": "01 Evaluation", "detail": "LLM-Fehler + Metriken + Tuning", "icon": "📐", "color": "#0078d4"},
    {"label": "02 Optimierung", "detail": "Manuell vs. automatisch", "icon": "⚙️", "color": "#ca5010"},
    {"label": "03 Domain-Daten", "detail": "Dein Burggraben", "icon": "🏰", "color": "#ca5010"},
    {"label": "04 Agenten", "detail": "Tool-Nutzung", "icon": "🤖", "color": "#107c10"},
    {"label": "05 Gesamtbild", "detail": "Showdown + Quiz", "icon": "🎯", "color": "#107c10"},
], title="Dein Lernpfad")


## ⚙️ Modell konfigurieren

Wir verwenden ein voreingestelltes Modell. Du kannst es in der nächsten Zelle ändern.


In [ ]:
from dspy_tasks.config import get_available_models, configure_dspy

# Verfügbare Modelle
print("Verfügbare Modelle:")
for m in get_available_models()[:10]:
    print(f"  • {m}")

# Modell wählen (ändere den String um ein anderes zu nutzen)
MODEL = "github_copilot/gpt-5.1"
configure_dspy(MODEL)
print(f"\n✅ Konfiguriert: {MODEL}")


## 1. Wie gut ist das Modell wirklich?

Lass uns das Modell auf **Fragen mit bekannter Antwort** testen. Wir schauen genau hin: wo liegt es richtig — und wo **daneben**?


In [ ]:
import dspy
from dspy_tasks.config import configure_dspy

configure_dspy(model=MODEL)

# Fragen mit BEKANNTER richtiger Antwort
test_questions = [
    ("Was ist die Hauptstadt von Australien?",  "Canberra"),
    ("Wie viele Planeten hat unser Sonnensystem?", "8"),
    ("Ist Glas eine Flüssigkeit?", "Nein"),
    ("Können Goldische nur 3 Sekunden erinnern?", "Nein"),
    ("Sieht man die Chinesische Mauer vom Weltraum?", "Nein"),
    ("Was ist schwerer: 1kg Stahl oder 1kg Federn?", "Gleich schwer"),
    ("Wie viel Prozent des Gehirns nutzen Menschen?", "100%"),
]

class QA(dspy.Signature):
    """Beantworte die Frage kurz und korrekt."""
    question = dspy.InputField(desc="Eine Wissensfrage")
    answer = dspy.OutputField(desc="Kurze, korrekte Antwort")

qa = dspy.Predict(QA)

print("Frage | Erwartete Antwort | Modell-Antwort | Match?")
print("=" * 80)

for question, expected in test_questions:
    result = qa(question=question)
    model_answer = result.answer.strip()
    exact = expected.lower() == model_answer.lower()
    contains = expected.lower() in model_answer.lower()
    
    if exact:
        icon = "✅ Exakt"
    elif contains:
        icon = "🟡 Enthalten"
    else:
        icon = "❓ Unklar"
    
    print(f"{icon}")
    print(f"   Frage:    {question}")
    print(f"   Erwartet: {expected}")
    print(f"   Modell:   {model_answer[:80]}")
    print()


## 💡 Siehst du das Problem?

Schau dir die Ergebnisse oben genau an. Das Modell antwortet zum Beispiel:

- Wir erwarten **"8"** → Modell sagt **"Acht."** — Ist das richtig? Ja! Aber ein einfacher String-Vergleich sagt: ❌
- Wir erwarten **"Gleich schwer"** → Modell sagt **"Beides wiegt gleich viel"** — Inhaltlich korrekt, aber komplett anderer Text!
- Wir erwarten **"Nein"** → Modell sagt **"Nein, Glas ist ein amorpher Feststoff..."** — Richtig, aber viel zu lang

**Das ist das fundamentale Problem:** LLMs geben jedes Mal einen **anderen String** zurück — selbst wenn die Antwort inhaltlich stimmt. Wie willst du da automatisch messen, ob die Antwort richtig ist?

Genau dafür brauchst du **Metriken** — clevere Funktionen die nicht nur `==` vergleichen, sondern verstehen, ob die *Bedeutung* stimmt.

Das ist das Thema der nächsten Sektion. 👇


## 2. Wie kann man LLM-Antworten automatisch bewerten?

Das Problem hast du gesehen: das Modell sagt "Acht" statt "8" — inhaltlich richtig, aber ein String-Vergleich scheitert. Hier sind 5 Strategien, wie man das löst:


## 🔧 Wie löst man das? — 5 Strategien

### 1. Antwort einschränken
Statt dem Modell freien Text zu erlauben, gibst du vor: **"Antworte nur mit Ja oder Nein"** oder **"Antworte mit einer Zahl"**. Dann reicht ein einfacher Vergleich.

### 2. Multiple Choice
Du gibst die möglichen Antworten vor: **A, B, C oder D**. Das Modell muss einen Buchstaben wählen — leicht zu prüfen.

### 3. Schlüsselwörter suchen
Statt exaktem Match prüfst du, ob bestimmte Schlüsselwörter in der Antwort vorkommen. "Canberra" in "Die Hauptstadt ist Canberra" → ✅

### 4. Semantischer Vergleich
Du nutzt Embeddings oder eine Ähnlichkeits-Metrik (z.B. Token-F1) um zu messen, wie *ähnlich* die Antwort der erwarteten ist — auch wenn die Worte anders sind.

### 5. Ein LLM als Richter (LLM-as-Judge)
Du lässt ein **zweites LLM** bewerten: *"Sagt diese Antwort inhaltlich das Gleiche wie die Referenz?"* Das klingt verrückt, funktioniert aber erstaunlich gut.


### Strategie 1 ausprobieren: Antworten einschränken

Wenn wir dem Modell sagen "antworte nur mit einem Wort", wird der Vergleich trivial:


In [ ]:
# Lass uns Strategie 1 und 5 direkt ausprobieren!

# --- Strategie 1: Antwort einschränken ---
class StrictQA(dspy.Signature):
    """Beantworte die Frage. Antworte NUR mit einer Zahl, 'Ja', 'Nein', oder einem einzelnen Wort."""
    question = dspy.InputField(desc="Eine Wissensfrage")
    answer = dspy.OutputField(desc="Nur ein Wort oder eine Zahl, nichts sonst")

strict_qa = dspy.Predict(StrictQA)

print("━" * 50)
print("Strategie 1: Eingeschränkte Antworten")
print("━" * 50)

test_pairs = [
    ("Wie viele Planeten hat unser Sonnensystem?", "8"),
    ("Ist Glas eine Flüssigkeit?", "nein"),
    ("Was ist die Hauptstadt von Australien?", "canberra"),
]

for q, expected in test_pairs:
    result = strict_qa(question=q)
    answer = result.answer.strip().lower().rstrip('.')
    match = answer == expected.lower()
    icon = "✅" if match else "❌"
    print(f"  {icon} {q}")
    print(f"     Modell: '{result.answer.strip()}' | Erwartet: '{expected}' | Exakt: {match}")
    print()

print("👆 Viel einfacher zu vergleichen wenn die Antwort eingeschränkt ist!")


### Strategie 5 ausprobieren: LLM als Richter

Ein zweites LLM bewertet, ob die Antwort **inhaltlich** korrekt ist — auch wenn die Worte ganz anders sind:


In [ ]:
# --- Strategie 5: LLM-as-Judge ---
class Judge(dspy.Signature):
    """Bewerte ob die gegebene Antwort inhaltlich korrekt ist. Antworte NUR mit 'korrekt' oder 'falsch'."""
    question = dspy.InputField(desc="Die ursprüngliche Frage")
    expected_answer = dspy.InputField(desc="Die Referenz-Antwort")
    model_answer = dspy.InputField(desc="Die zu bewertende Antwort")
    verdict = dspy.OutputField(desc="Nur 'korrekt' oder 'falsch'")

judge = dspy.Predict(Judge)

print("━" * 50)
print("Strategie 5: LLM als Richter")
print("━" * 50)

judge_cases = [
    ("Wie viele Planeten?", "8", "Acht."),
    ("Was ist schwerer: 1kg Stahl oder 1kg Federn?", "Gleich schwer", "Beides wiegt gleich viel."),
    ("Gehirn-Nutzung?", "100%", "Menschen nutzen nahezu alle Teile ihres Gehirns."),
    ("Hauptstadt Australien?", "Canberra", "Sydney"),
]

for q, expected, model_ans in judge_cases:
    result = judge(question=q, expected_answer=expected, model_answer=model_ans)
    verdict = result.verdict.strip().lower()
    icon = "✅" if "korrekt" in verdict else "❌"
    print(f"  {icon} Frage: {q}")
    print(f"     Erwartet: '{expected}' | Modell sagte: '{model_ans}'")
    print(f"     Richter-Urteil: {result.verdict.strip()}")
    print()

print("👆 Der LLM-Richter versteht, dass 'Acht' == '8' und 'Beides wiegt gleich' == 'Gleich schwer'!")
print("   Aber: er kann sich auch irren — und er kostet einen extra API-Aufruf pro Bewertung.")


### 📊 Jede Strategie hat Tradeoffs

| Strategie | Vorteile | Nachteile |
|-----------|----------|----------|
| Antwort einschränken | Einfach, schnell, deterministisch | Geht nur bei geschlossenen Fragen |
| Multiple Choice | Leicht zu prüfen | Muss Optionen vorgeben |
| Schlüsselwörter | Flexibler als exakt | Kann false positives haben |
| Semantischer Vergleich | Erkennt Paraphrasen | Braucht Embeddings/Berechnung |
| LLM-as-Judge | Versteht Bedeutung | Kostet extra, kann sich auch irren |

In der Praxis kombiniert man diese Strategien — je nach Aufgabe. Genau das machen die **Metrik-Funktionen**, die wir gleich kennenlernen.


## 3. Metriken in der Praxis

In der Praxis nutzt man verschiedene Metriken je nach Aufgabe — von simplem Exact Match bis hin zu gewichteten Composite-Scores.


In [ ]:
from dspy_tasks.visualize import diagram_compare

diagram_compare(
    {"title": "Klassische Software", "items": ["Unit Test", "assert x == y", "Pass/Fail"], "icon": "🔧", "color": "#8a8886"},
    {"title": "KI-Software", "items": ["Metrik-Funktion", "metric(gold, pred) → score", "0.0 bis 1.0"], "icon": "🧠", "color": "#0078d4"},
    title="Tests vs. Metriken"
)

### Die verschiedenen Metrik-Level

Metriken werden immer raffinierter. Hier siehst du drei Level — von einfach bis komplex:


In [ ]:
from dspy_tasks.calculations import token_f1

print("━" * 50)
print("📏 Level 1: Exact Match")
print("  Der einfachste Check: stimmt die Antwort exakt?")
print(f"  'positive' == 'positive' → 1.0 ✅")
print(f"  'positive' == 'negative' → 0.0 ❌")
print()
print("━" * 50)
print("📏 Level 2: Token F1 (Partial Credit)")
print("  Was wenn die Antwort TEILWEISE stimmt?")
print("  F1 balanciert Precision (wie viele Treffer waren korrekt?)")
print("  und Recall (wie viele wurden gefunden?)")
print()

cases = [
    (["apple", "banana", "cherry"], ["apple"],
     "Vorsichtig: nur 1 von 3 gefunden, aber der war richtig"),
    (["apple", "banana", "cherry"], ["apple", "banana", "cherry", "grape", "melon"],
     "Übereifrig: alles gefunden, aber 2 Falsche dabei"),
    (["apple", "banana", "cherry"], ["apple", "banana", "cherry"],
     "Perfekt: genau richtig"),
    (["apple", "banana"], ["cherry", "grape"],
     "Komplett daneben: nichts stimmt"),
]
for gold, pred, desc in cases:
    f1 = token_f1(gold, pred)
    icon = "✅" if f1 == 1.0 else "🟡" if f1 > 0 else "❌"
    print(f"  {icon} {desc}")
    print(f"     Gold: {gold} | Pred: {pred} → F1 = {f1:.2f}")

print()
print("━" * 50)
print("📏 Level 3: Composite (Ticket Routing)")
print("  Mehrere Kriterien mit Gewichtung:")
print("  Priorität korrekt (40%) + Kategorie (35%) + Team (25%)")
print("  = Gewichtete Spezifikation von 'was am wichtigsten ist'")


## 4. Kann ich mit dem Prompt die Genauigkeit verbessern?

Du hast gesehen, dass das Modell Fehler macht. Und du weisst jetzt, wie man Genauigkeit misst. Die naheliegende Frage: **hilft eine bessere Formulierung?**

Hier testen wir **Ticket-Routing**: Das Modell muss IT-Tickets nach 3 Feldern klassifizieren:

| Feld | Gewicht | Mögliche Werte |
|------|---------|----------------|
| **Priorität** | 40% | Standard, Medium, High |
| **Kategorie** | 35% | Service Request, Failure, Event NO Customer Impact |
| **Gruppe** | 25% | 21 verschiedene Teams (z.B. "SDE - Service Desk", "OFC - Office & Collaboration", ...) |

Der Score ist der gewichtete Durchschnitt: jedes richtige Feld gibt seinen Anteil. Ohne die korrekten Werte zu kennen, wird das Modell daneben liegen — genau das wollen wir zeigen!

In [ ]:
import json, pandas as pd
from IPython.display import display, HTML
from dspy_tasks.actions import run_with_prompt
from dspy_tasks.visualize import display_score, display_results_table

# Daten laden und gültige Werte extrahieren
with open("datasets/ticket_routing.json") as f:
    tickets = json.load(f)

categories = sorted(set(t["category"] for t in tickets))
priorities = sorted(set(t["priority"] for t in tickets))
groups = sorted(set(t["assigned_group"] for t in tickets))

display(HTML("""
<div style="border-left:4px solid #0078d4; padding:12px; margin-bottom:12px; border-radius:0 8px 8px 0">
    <b>📂 Datensatz:</b> <a href="datasets/ticket_routing.json">datasets/ticket_routing.json</a>
    &nbsp;|&nbsp; <b>{total} Tickets</b>
    &nbsp;|&nbsp; <b>{n_cats} Kategorien</b>
    &nbsp;|&nbsp; <b>{n_prios} Prioritäten</b>
    &nbsp;|&nbsp; <b>{n_groups} Gruppen</b>
</div>
""".format(total=len(tickets), n_cats=len(categories), n_prios=len(priorities), n_groups=len(groups))))

display(pd.DataFrame(tickets[:6]))

# --- Versuch 1: Generischer Prompt (kennt die gültigen Werte NICHT) ---
PROMPT_V1 = "Classify this IT support ticket by category, priority, and assigned group."

# --- Versuch 2: Prompt MIT den gültigen Werten ---
PROMPT_V2 = f"""Classify this IT support ticket.
Category MUST be one of: {', '.join(categories)}
Priority MUST be one of: {', '.join(priorities)}
Assigned group MUST be one of: {', '.join(groups)}"""

for label, prompt in [("Generisch (ohne Optionen)", PROMPT_V1), ("Mit gültigen Werten", PROMPT_V2)]:
    print(f"\n{'='*60}")
    print(f"📝 {label}")
    print(f"   {prompt[:100]}...")
    result = run_with_prompt("ticket_routing", prompt, max_eval=4)
    display_score(label, result.score)
    display_results_table(result.individual_scores)

### 💡 Und — hat es geholfen?

- Wurde dein Score besser?
- Wie viele Versuche hast du gebraucht?
- Hättest du das für 20 verschiedene Aufgaben machen wollen?

Genau das ist das Problem mit manuellem Prompt-Tuning: es **funktioniert** — aber es ist **langsam, mühsam und fragil**. Was bei einer Aufgabe hilft, verschlechtert vielleicht eine andere.

> Im nächsten Notebook sehen wir, wie man das **automatisiert**.


## 5. Kann das Modell Code schreiben?

Eine besonders spannende Aufgabe: das Modell soll **Python-Code generieren**. Hier wird Evaluation richtig interessant — denn wir können den generierten Code tatsächlich **ausführen** und mit echten Testdaten prüfen ob er **korrekte Ergebnisse** liefert!

Das ist viel aussagekräftiger als nur zu checken ob der Code kompiliert. Denn Code der läuft aber falsche Ergebnisse liefert ist nutzlos. Wir testen jede generierte Funktion mit konkreten Eingaben und vergleichen die Ausgabe mit dem erwarteten Resultat.

In [ ]:
import re

# Für Code-Generierung testen wir absichtlich ein KLEINERES Modell
# um zu zeigen dass nicht jedes Modell alles kann
SMALL_MODEL = "github_copilot/gpt-4o-mini"
configure_dspy(SMALL_MODEL)
print(f"⚠️ Für diesen Test nutzen wir absichtlich ein kleineres Modell: {SMALL_MODEL}")
print()

task = get_task("code_generation")
print(f"📋 Task: {task.name}")
print(f"   Metrik: Echte Ausführung mit Testdaten!")
print()

# Testdaten pro REFERENZ-Funktion — wir matchen über den Referenz-Namen, nicht den generierten
TEST_INPUTS = {
    "fibonacci": [[0], [1], [10], [20]],
    "is_palindrome": [["racecar"], ["hello"], ["A man a plan a canal Panama"]],
    "flatten": [[[1, [2], 3]], [[1, [2, [3, [4]]]]]],
    "word_frequency": [["the cat sat on the mat"]],
    "remove_duplicates": [[[1, 2, 3, 2, 1, 4]]],
    "eval_expr": [
        ["3 + 4 * 2"],           # = 11.0 (nicht 14!)
        ["(1 + 2) * (3 + 4)"],   # = 21.0
        ["10 / 3"],              # = 3.333...
        ["2 * 3 + 4 * 5"],       # = 26.0
        ["(2 + 3) * (7 - 4) / 5"], # = 3.0
        ["-3 + 5"],              # = 2.0
    ],
    "merge_intervals": [
        [[[1,3],[2,6],[8,10],[15,18]]],       # → [[1,6],[8,10],[15,18]]
        [[[1,4],[4,5]]],                       # → [[1,5]] (touching)
        [[[1,4],[2,3]]],                       # → [[1,4]] (fully contained)
        [[[1,10],[2,3],[4,5],[6,7],[8,9]]],    # → [[1,10]]
        [[]],                                   # → []
    ],
}

# Erste 5 einfache + die 2 schweren am Ende
examples = task.load_examples()[:5] + task.load_examples()[-2:]
module = task.make_module()

total_tasks = len(examples)
tasks_passed = 0
tasks_failed = 0
tasks_error = 0

for ex in examples:
    desc = str(ex.description)[:80]
    print(f"📝 Aufgabe: {desc}")
    prediction = module(description=ex.description)
    code = prediction.python_code.strip()
    
    # Markdown-Codeblöcke entfernen (```python ... ```)
    code = re.sub(r'^```\w*\n?', '', code)
    code = re.sub(r'\n?```$', '', code)
    
    # Code anzeigen
    print(f"   Generierter Code:")
    for line in code.split('\n')[:8]:
        print(f"   │ {line}")
    if code.count('\n') > 7:
        print(f"   │ ... ({code.count(chr(10))+1} Zeilen)")
    
    # Generierte Funktion ausführen
    gen_ns = {}
    try:
        import signal
        def _timeout_handler(signum, frame):
            raise TimeoutError('Code execution took too long (possible infinite loop)')
        old_handler = signal.signal(signal.SIGALRM, _timeout_handler)
        signal.alarm(5)  # 5 second timeout
        exec(compile(code, '<generated>', 'exec'), gen_ns)
        signal.alarm(0)  # cancel timeout
        signal.signal(signal.SIGALRM, old_handler)
    except TimeoutError:
        signal.alarm(0)
        print(f"   ❌ Timeout: Code läuft zu lange (Endlosschleife?)")
        tasks_error += 1
        print()
        continue
    except Exception as e:
        signal.alarm(0)
        print(f"   ❌ Kompilierung fehlgeschlagen: {type(e).__name__}: {e}")
        tasks_error += 1
        print()
        continue
    
    # Referenz-Funktion aus dem Datensatz laden
    ref_ns = {}
    exec(compile(str(ex.python_code), '<reference>', 'exec'), ref_ns)
    
    # Referenz-Funktion finden (Name aus dem Datensatz)
    ref_func, ref_name = None, None
    for name, obj in ref_ns.items():
        if callable(obj) and not name.startswith('_'):
            ref_func, ref_name = obj, name
            break
    
    # Generierte Funktion finden
    gen_func = None
    for name, obj in gen_ns.items():
        if callable(obj) and not name.startswith('_'):
            gen_func = obj
            break
    
    if gen_func is None or ref_func is None:
        print(f"   ❌ Funktion nicht gefunden")
        tasks_error += 1
        print()
        continue
    
    # Testdaten über REFERENZ-Namen holen (nicht den generierten Namen)
    test_inputs = TEST_INPUTS.get(ref_name, [])
    if not test_inputs:
        print(f"   ❌ Keine Testdaten für '{ref_name}'")
        tasks_error += 1
        print()
        continue
    
    # Jeden Testfall ausführen: generiert vs. Referenz
    task_ok = True
    for args in test_inputs:
        try:
            signal.signal(signal.SIGALRM, _timeout_handler)
            signal.alarm(5)
            expected = ref_func(*args)
            got = gen_func(*args)
            signal.alarm(0)
            # Für Floats: Toleranz-Vergleich
            if isinstance(expected, float) and isinstance(got, (int, float)):
                match = abs(float(got) - expected) < 1e-6
            else:
                match = got == expected
            if match:
                print(f"   ✅ {ref_name}({', '.join(repr(a) for a in args)}) → {repr(got)}")
            else:
                task_ok = False
                print(f"   ❌ {ref_name}({', '.join(repr(a) for a in args)})")
                print(f"      Erwartet: {repr(expected)}")
                print(f"      Bekommen: {repr(got)}")
        except Exception as e:
            task_ok = False
            print(f"   💥 {ref_name}({', '.join(repr(a) for a in args)}) → {type(e).__name__}: {e}")
    
    if task_ok:
        tasks_passed += 1
    else:
        tasks_failed += 1
    print()

print("━" * 60)
print(f"📊 Ergebnis: {tasks_passed}/{total_tasks} Aufgaben bestanden")
if tasks_failed:
    print(f"   ❌ {tasks_failed} Aufgaben mit falschen Ergebnissen")
if tasks_error:
    print(f"   💥 {tasks_error} Aufgaben mit Fehlern (Kompilierung, keine Funktion, etc.)")
print()
print("💡 Siehst du den Unterschied?")
print("   Ob Code KOMPILIERT sagt wenig aus.")
print("   Ob er die RICHTIGEN ERGEBNISSE liefert — das zählt!")
print()
print("   Das ist wie bei Software-Tests:")
print("   Ein Programm das startet aber falsch rechnet ist wertlos.")

# Zurück zum Haupt-Modell für die nächsten Zellen
configure_dspy(MODEL)

## ⏭️ Weiter geht's!

Du hast Metriken, du kannst messen, du hast manuell getuned. Beim Ticket-Routing hast du gesehen: selbst mit allen gültigen Werten im Prompt bleibt der Score bescheiden.

Was, wenn der **Computer die Prompts SELBST optimieren** könnte — und dabei aus den Trainingsdaten die Zuordnungsregeln lernt?

👉 **[Weiter zu Notebook: Automatische Optimierung →](02_optimization.ipynb)**

---

## 🧪 EXTRA: Warum ist Evaluation so schwer?

**TruthfulQA** ist ein Benchmark aus der KI-Forschung — Fragen, bei denen LLMs besonders gerne falsch antworten.

Aber hier steckt ein Twist: Das Modell antwortet oft **inhaltlich korrekt**, aber die Metrik sagt trotzdem **0%**. Warum?

Schau dir die Ergebnisse unten an und überlege:

1. **Welche der 5 Strategien** (von oben) würde hier am besten funktionieren?
2. **Warum wird selbst die beste Strategie nicht 100% erreichen?**

In [ ]:
from dspy_tasks.benchmarks import load_truthfulqa, contains_match
from dspy_tasks.actions import run_on_examples
import dspy

truthful_examples = load_truthfulqa(8)

class TQASig(dspy.Signature):
    """Placeholder"""
    question = dspy.InputField(desc="A factual question")
    answer = dspy.OutputField(desc="A truthful answer")

# Editiere diesen Prompt und führe die Zelle erneut aus!
TQA_PROMPT = "Answer the question accurately. Be careful about common misconceptions and myths. If the common belief is wrong, give the scientifically correct answer."

result = run_on_examples(truthful_examples, TQA_PROMPT, TQASig, contains_match)
display_score("TruthfulQA (contains_match)", result.score)
display_results_table(result.individual_scores)

print()
print("━" * 60)
print("🤔 Denkaufgabe:")
print()
print("Die Metrik 'contains_match' prüft: Ist die Gold-Antwort")
print("als Substring in der Modell-Antwort enthalten?")
print()
print("z.B. Gold: 'No, goldfish can remember things for months.'")
print("     Modell: 'No. The 3-second memory idea is a myth...'")
print("     → 0% weil der exakte Satz nicht vorkommt!")
print()
print("Welche Strategie wäre besser?")
print("  → LLM-as-Judge (Strategie 5) würde ~90% erreichen")
print("  → Aber auch ein LLM-Richter ist nicht perfekt:")
print("    • Er kann sich irren (selten, aber möglich)")
print("    • Er kostet einen extra API-Aufruf pro Beispiel")
print("    • Sein Urteil kann je nach Modell variieren")
print()
print("💡 Fazit: Es gibt keine perfekte Metrik.")
print("   Die Kunst ist, die RICHTIGE Metrik für die Aufgabe zu wählen.")